# 04 — AML over the transfer graph (pragmatiq's own extension)

> **This notebook is NOT part of the PRAGMA paper.** The graph-based AML work here is
> pragmatiq's own extension, built standalone on top of the paper's user embeddings. It
> explores a question the paper does not: can a graph neural network over a money-transfer
> graph recover signal that a per-user embedding alone cannot see? Everything in notebooks
> 01–03 is the paper's recipe; this is where we go beyond it.

Mule-ring (money-laundering) detection is a **relational** problem: a launderer is
defined by *who they transact with* (fan-in of small credits, layering among the
ring, rapid fan-out), not just their own behaviour. A per-user embedding — however
good — cannot by construction see a counterparty pattern that spans accounts. The
synthetic mule rings here are **multi-hop layered laundering chains**: mule amounts
and counterparty degree are drawn to *match ordinary accounts*, so 1-hop degree is
not a trivial oracle. So we build a transfer graph from `transfers.parquet` and run a
four-arm comparison:

- **(a) isolated pragmatiq embeddings** — a probe on each user's embedding alone. No
  graph, so on a relational signal it should clearly underperform.
- **(b) GraphSAGE + pragmatiq features** — message passing over the transfer graph with
  pragmatiq embeddings as node features. Tests whether the learned per-user embedding
  picks up the relational signal.
- **(c) GraphSAGE + hand-crafted features** — the same graph, but generic structural
  node features (degree, volume, counterparties).
- **(d) logistic control** — the same hand-crafted features, no graph. Tells us whether
  the graph adds over the features themselves.

The GraphSAGE model and this ablation live in `pragmatiq/models/gnn.py` and need the
optional `.[gnn]` extra (`pip install -e ".[gnn]"`). The model architecture itself
(notebooks 01–03) does not depend on any of this.

> pragmatiq is an independent implementation inspired by the PRAGMA paper
> (arXiv 2604.08649) and is not affiliated with or endorsed by Revolut.

## Setup

Keep imports, constants, and synthetic-data settings at the top. The AML run is
larger than the quickstart notebooks, so adjust the config before executing if
you are running on a small CPU machine.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from pragmatiq import api

WORK = Path("runs_aml")
SYNTH_CONFIG = {
    "n_users": 12000,
    "months": 16,
    "n_merchants": 4000,
    "mule_ring_count": 80,
    "seed": 5,
    "eval_month_credit": 4,
    "eval_month_short": 10,
}
TOKENIZER_CONFIG = {
    "target_vocab": 28000,
    "n_buckets": 64,
}
PRETRAIN_CONFIG = {
    "max_steps": 2000,
    "token_budget": 16384,
}
ABLATION_SEEDS = (0, 1, 2)

## Build the AML run

Generate mule-ring data, tokenize it, and pretrain a small model before running
the graph ablation.

In [ ]:
api.synthesize(
    SYNTH_CONFIG,
    out=WORK / "raw",
    n_workers=8,
    write_report=False,
)
api.tokenize(WORK / "raw", WORK / "tok", config=TOKENIZER_CONFIG)

summary = api.pretrain(
    WORK / "tok",
    "aml",
    model_size="small",
    config=PRETRAIN_CONFIG,
    runs_root=WORK / "runs",
)

display(summary)

## Run the four-arm ablation

The table compares an isolated probe, GraphSAGE with pragmatiq embeddings, GraphSAGE
with hand-crafted structural features over the same transfer graph, and a logistic
control on those hand-crafted features without the graph.

In [ ]:
result = api.gnn(
    WORK / "tok",
    summary["run_dir"],
    WORK / "raw" / "transfers.parquet",
    WORK / "raw" / "labels" / "aml.parquet",
    seeds=ABLATION_SEEDS,
    epochs=150,
)

table = pd.DataFrame(
    {
        setup: {"AUC mean": values["mean"], "AUC std": values["std"]}
        for setup, values in result["per_setup"].items()
    }
).T

print("verdict:", result["verdict"])
display(table)

## Result

**The gated claim — relational recovery (true and robust).** A GraphSAGE over the
transfer graph recovers money-mule rings that a probe on the isolated per-user
embedding cannot: **(c) ≫ (a)** by a clear margin, so the AML signal lives in the
multi-hop transfer structure an isolated embedding misses. Message passing adds over
the same features without a graph (**(c) > (d)**). The mules are multi-hop layered
laundering chains whose amounts and counterparty degree match ordinary accounts, so
1-hop degree is not a trivial oracle (the degree-only control (d) is only moderate);
the discriminative signal is the multi-hop layering chain. This is what `gate_6`
gates, at both CI and full scale.

**Reported, not gated — the honest limitation.** The learned per-user embedding adds
only a little over the isolated probe (**(b) > (a)**) and does **not** beat
hand-crafted features (**(b) < (c)**). The isolated embedding sits near chance, so the
model does not capture the multi-hop laundering signal in the per-user representation;
recovering it in a learned representation is the **open challenge**. This is consistent
with the PRAGMA paper's own observation that AML is a setting where the model
underperforms because it processes user histories in isolation — the GNN here probes
that gap rather than claiming the learned embedding wins.

The auto-generated results table below (written by `gate_6` / `write_aml_report`) shows
the actual numbers for this run. See `MODEL_CARD.md` for the full discussion.

## Latest ablation result

| setup | ROC-AUC (mean ± std over seeds) |
| --- | --- |
| (a) probe on isolated pragmatiq embeddings | 0.498 ± 0.010 |
| (b) GraphSAGE over transfers + pragmatiq features | 0.554 ± 0.026 |
| (c) GraphSAGE + hand-crafted node features | 0.670 ± 0.014 |
| (d) control: logistic regression on the same hand-crafted features, no graph | 0.604 ± 0.028 |

**Relational recovery (gated): True** — a GraphSAGE over the transfer graph recovers money-mule rings that a probe on isolated pragmatiq embeddings cannot ((c) > (a) = True), so the AML signal lives in the multi-hop transfer structure an isolated per-user embedding misses. Money mules are degree- and volume-matched to ordinary accounts, so the signal is the multi-hop layering chain, not 1-hop degree, and message passing adds over the same features without a graph ((c) > (d) = True). The gate requires both.

**Reported, not gated:** the learned per-user embedding adds a little over the isolated probe ((b) > (a) = True) but does not beat hand-crafted features ((b) > (c) = False). The isolated embedding sits near chance, so on this synthetic book the model does not capture the multi-hop laundering signal on its own — recovering it in a learned per-user representation is the open challenge (see MODEL_CARD.md).

<sub>provenance: n_nodes=12000, n_edges=344388, n_mules=607, seeds=[0, 1, 2], epochs=150, commit=a263737</sub>
